# Ch 24 — LoRA · QLoRA 입문

LoRA 의 핵심 식과 HuggingFace `peft` 30줄로 Qwen 2.5-0.5B-Instruct 에 LoRA SFT. 안전한 기본값: r=16, alpha=32, target=q,v,k,o, lr=1e-4.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part7/ch24_lora_intro.ipynb)

In [ ]:
# !pip install -q transformers peft datasets bitsandbytes accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. 컨셉 — Low-Rank 가 충분한 이유

LoRA (Hu et al., 2021) 의 핵심 가설:

> "파인튜닝 시의 가중치 변화 ΔW 는 **low-rank 행렬** 로 잘 근사된다."

표준 가중치 갱신:  `W' = W + ΔW`

LoRA 의 가정:  `ΔW = B·A`,  `A ∈ R^(r×d)`,  `B ∈ R^(d×r)`

즉 ΔW 를 **두 작은 행렬의 곱** 으로 분해. r 이 작으면 (예: 8, 16) 학습 파라미터가 W 의 0.1~1%.

## 3. peft 30줄 LoRA SFT (최소 예제)

In [ ]:
# lora_sft.py
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

base = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.bfloat16, device_map="auto")

# 1. LoRA config — 안전한 기본값
lora_cfg = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
# trainable params: 4,358,144 || all params: 498,310,656 || trainable%: 0.87

In [ ]:
# 2. 데이터 — instruction 페어 (domain_pairs.jsonl 있다고 가정)
# 없으면 간단한 더미 데이터로 테스트
import json, os

if not os.path.exists("domain_pairs.jsonl"):
    dummy = [
        {"instruction": "3~5세 어린이용 한국어 동화 한 편을 만들어줘.",
         "output": "옛날 옛날에 작은 토끼가 살았어요. 토끼는 당근을 좋아했지요."},
        {"instruction": "짧은 동화를 써줘.",
         "output": "숲속에 작은 곰이 살았어요. 곰은 꿀을 찾아 매일 산을 올랐어요."},
    ] * 50  # 100 더미 페어
    with open("domain_pairs.jsonl", "w") as f:
        for d in dummy: f.write(json.dumps(d, ensure_ascii=False) + "\n")
    print("더미 domain_pairs.jsonl 생성")

def format_pair(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    text = tok.apply_chat_template(msgs, tokenize=False)  # chat template 적용
    return tok(text, truncation=True, max_length=512, padding="max_length", return_tensors="pt")

ds = load_dataset("json", data_files="domain_pairs.jsonl")["train"]
ds = ds.map(format_pair).remove_columns(["instruction", "output"])
print(f"데이터셋 크기: {len(ds)} 페어")

In [ ]:
# 3. Trainer
args = TrainingArguments(
    output_dir="lora_out",
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    num_train_epochs=3, learning_rate=1e-4,
    warmup_steps=20, lr_scheduler_type="cosine",
    bf16=True, logging_steps=10, save_steps=200,
)
trainer = Trainer(model=model, args=args, train_dataset=ds, tokenizer=tok)
trainer.train()
model.save_pretrained("lora_out/adapter")  # 어댑터만 저장 (~20MB)

## 4. QLoRA — 4bit 베이스로 메모리 1/4 (실전)

In [ ]:
# qlora.py
from transformers import BitsAndBytesConfig, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import torch

base = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat 4 (Dettmers 2023)
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,     # 추가 압축
)
model_q = AutoModelForCausalLM.from_pretrained(base, quantization_config=bnb_cfg, device_map="auto")

# 그 다음은 LoRA 와 동일
lora_cfg = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model_q = get_peft_model(model_q, lora_cfg)
model_q.print_trainable_parameters()

## 6. 학습 후 — 어댑터 합치기 vs 분리 유지

In [ ]:
# merge_or_keep.py
from peft import PeftModel
from transformers import AutoModelForCausalLM
import torch

base = "Qwen/Qwen2.5-0.5B-Instruct"

# 옵션 1: 분리 유지 (배포 시 base + adapter)
base_model = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.bfloat16)
model_with_adapter = PeftModel.from_pretrained(base_model, "lora_out/adapter")
print("어댑터 로드 완료 (분리 유지)")

# 옵션 2: 합쳐서 단일 모델로
merged = model_with_adapter.merge_and_unload()
merged.save_pretrained("merged_model")
print("합치기 + 저장 완료 (merged_model/)")

## 연습문제

1. SmolLM2-135M 에 100 페어 LoRA. 학습 전·후 PPL 차이는?
2. r=4 / 16 / 64 로 같은 데이터 학습. 학습 손실 + 시간 + 어댑터 크기 비교.
3. `target_modules=["q_proj","v_proj"]` vs `["q_proj","k_proj","v_proj","o_proj"]` 비교.
4. QLoRA (nf4) 와 일반 LoRA (bf16) 의 학습 속도 + 최종 손실 차이.
5. **(생각해볼 것)** "low-rank 가 충분하다" 는 가설이 깨지는 도메인은? r=64 도 부족한 작업이 있을까?

In [ ]:
# 연습 1: SmolLM2-135M LoRA 학습 전/후 PPL


In [ ]:
# 연습 2: r=4 / 16 / 64 비교


In [ ]:
# 연습 3: target_modules 비교


In [ ]:
# 연습 4: QLoRA vs LoRA 비교
